# **configuração do ambiente**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import unidecode


In [ ]:
%pip install unidecode

In [ ]:
# Configurar visualização
sns.set(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.figsize"] = (10,6)


In [ ]:
# Verificar versão do pandas (só para garantir compatibilidade)
import pandas as pd
print("Versão do pandas:", pd.__version__)


# **Upload das bases no Colab**

In [ ]:
#from google.colab import drive
#drive.mount('/content/drive')

In [ ]:
# Leitura das bases
base_tea_tdah = pd.read_csv('data/base_tea_tdah_10000.csv')
saeb_2ef = pd.read_csv('data/SAEB_2EF_tratado.csv')
saeb_34em = pd.read_csv('data/SAEB_ALUNO_34EM_tratado.csv')
saeb_5ef = pd.read_csv('data/SAEB_ALUNO_5EF_tratado.csv')
saeb_9ef = pd.read_csv ('data/SAEB_ALUNO_9EF_tratado.csv')
saeb_escola = pd.read_csv('data/SAEB_escola_tratado.csv')
censo = pd.read_csv('data/censo_2023_tratado.csv')

print(" Bases carregadas com sucesso!")


In [ ]:
for nome, df in [('2EF', saeb_2ef), ('5EF', saeb_5ef), ('9EF', saeb_9ef), ('EM', saeb_34em), ('ESCOLA', saeb_escola)]:
    print(f"{nome}: {df.shape}")

# **limpando censo**

In [ ]:
# Seleção de colunas de importantes
colunas_interesse = [
    'NU_ANO_CENSO',
    'CO_MUNICIPIO',
    'CO_ENTIDADE',
    'NO_MUNICIPIO',
    'SG_UF',
    'NO_UF',
    'TP_DEPENDENCIA',
    'TP_LOCALIZACAO',
    'IN_BIBLIOTECA',
    'IN_LABORATORIO_INFORMATICA',
    'IN_INTERNET',
    'IN_INTERNET_ALUNOS',
    'IN_BANDA_LARGA'
]

censo_limpo = censo[colunas_interesse].copy()

In [ ]:
# Preenchimento de valores nulos

# Infraestrutura: 0 = não tem
infra_colunas = [
    'IN_BIBLIOTECA',
    'IN_LABORATORIO_INFORMATICA',
    'IN_INTERNET',
    'IN_INTERNET_ALUNOS',
    'IN_BANDA_LARGA'
]
censo_limpo[infra_colunas] = censo_limpo[infra_colunas].fillna(0)

# Categóricas: preencher com 'Não tem'
categoricas = ['TP_DEPENDENCIA', 'TP_LOCALIZACAO']
censo_limpo[categoricas] = censo_limpo[categoricas].fillna('Não tem')


In [ ]:
# 3. Garantir que binárias fiquem como 0/1

for col in infra_colunas:
    censo_limpo[col] = censo_limpo[col].apply(lambda x: 1 if x in [1, '1', True] else 0)


In [ ]:
# 4.Remover espaços extras de strings

censo_limpo = censo_limpo.applymap(lambda x: x.strip() if isinstance(x, str) else x)


In [ ]:
# 5. Remover duplicatas de escola

censo_limpo = censo_limpo.drop_duplicates(subset=['CO_ENTIDADE'], keep='first')



# **limpeza tdah e tea**

In [ ]:
pessoas = base_tea_tdah

#  2. Padronizar nomes de colunas
pessoas.columns = pessoas.columns.str.strip().str.upper()

#  3. Remover espaços extras nas colunas do tipo string
pessoas = pessoas.applymap(lambda x: x.strip() if isinstance(x, str) else x)

# 4. Corrigir tipos das colunas
pessoas['IDADE'] = pd.to_numeric(pessoas['IDADE'], errors='coerce')  # Força idade numérica
pessoas['RENDA'] = pd.to_numeric(pessoas['RENDA'], errors='coerce')  # Força renda numérica
pessoas['TDAH'] = pd.to_numeric(pessoas['TDAH'], errors='coerce').fillna(0).astype(int)
pessoas['TEA'] = pd.to_numeric(pessoas['TEA'], errors='coerce').fillna(0).astype(int)
pessoas['PESO_AMOSTRAL'] = pd.to_numeric(pessoas['PESO_AMOSTRAL'], errors='coerce').fillna(1.0)


#5. Padronizar colunas categóricas
# Sexo
pessoas['SEXO'] = pessoas['SEXO'].str.upper()
pessoas.loc[~pessoas['SEXO'].isin(['F', 'M']), 'SEXO'] = 'NAO_INFORMADO'

# Escolaridade
pessoas['ESCOLARIDADE'] = pessoas['ESCOLARIDADE'].str.title()
pessoas['ESCOLARIDADE'] = pessoas['ESCOLARIDADE'].fillna('Não informado')

# 7. Remover duplicatas pelo ID_PESSOA
pessoas = pessoas.drop_duplicates(subset=['ID_PESSOA'], keep='first')

#8. Resetar índice
pessoas = pessoas.reset_index(drop=True)

# Salvar a base tratada
pessoas.to_csv('results/base_tea_tdah_10000.csv', index=False)

print("Base de pessoas tratada com sucesso!")

# **limpeza SAEB 34EM**

In [ ]:
# 1. Padronizar nomes das colunas: sem espaços, uppercase
saeb_34em.columns = saeb_34em.columns.str.strip().str.upper().str.replace(' ', '_')

# 2. Substituir valores ausentes por NaN
saeb_34em.replace(['.', '', ' '], np.nan, inplace=True)

# 3. Ajustar tipos de colunas
id_cols = ['ID_ESCOLA', 'ID_ALUNO', 'ID_MUNICIPIO', 'ID_SERIE']
peso_cols = ['PESO_ALUNO_LP', 'PESO_ALUNO_MT', 'PESO_ALUNO_INSE']
prof_cols = [
    'PROFICIENCIA_LP', 'ERRO_PADRAO_LP', 'PROFICIENCIA_MT', 'ERRO_PADRAO_MT',
    'PROFICIENCIA_LP_SAEB', 'ERRO_PADRAO_LP_SAEB', 'PROFICIENCIA_MT_SAEB', 'ERRO_PADRAO_MT_SAEB'
]
question_cols = [col for col in saeb_34em.columns if col.startswith('TX_RESP_')]

# IDs como int
saeb_34em[id_cols] = saeb_34em[id_cols].astype('Int64')

# Pesos e proficiências como float
saeb_34em[peso_cols + prof_cols] = saeb_34em[peso_cols + prof_cols].astype('float')

# Respostas como string padronizada
saeb_34em[question_cols] = saeb_34em[question_cols].astype(str).applymap(lambda x: x.strip().upper() if pd.notna(x) else x)

# 4. Remover duplicatas de aluno na mesma série
saeb_34em = saeb_34em.drop_duplicates(subset=['ID_ALUNO', 'ID_SERIE'], keep='first')

# 5. Verificar se há colunas totalmente vazias e remover (opcional)
saeb_34em = saeb_34em.dropna(axis=1, how='all')

# Salvar a base tratada
saeb_34em.to_csv('results/SAEB_ALUNO_34EM_tratado.csv', index=False)

print("Base de respostas dos alunos limpa e pronta para merge!")


# **limpeza SAEB 2023 escola**

In [ ]:

# 1. Padronizar nomes das colunas
saeb_escola.columns = saeb_escola.columns.str.strip().str.upper().str.replace(' ', '_')

# 2. Substituir valores ausentes ou em branco por NaN
saeb_escola.replace(['', ' ', '.'], np.nan, inplace=True)

# 3. Identificar tipos de colunas
id_cols = [col for col in saeb_escola.columns if 'ID_' in col or col=='C']
taxa_cols = [col for col in saeb_escola.columns if 'TAXA' in col or 'NU_MATRICULADOS' in col or 'NU_PRESENTES' in col]
media_cols = [col for col in saeb_escola.columns if 'MEDIA' in col or 'NIVEL' in col or 'PC_' in col]
categorical_cols = [col for col in saeb_escola.columns if col not in id_cols+taxa_cols+media_cols]

# 4. Converter tipos
# IDs como inteiros
saeb_escola[id_cols] = saeb_escola[id_cols].astype('Int64')

# Taxas, médias e proficiências como float
saeb_escola[taxa_cols + media_cols] = saeb_escola[taxa_cols + media_cols].astype('float')

# Categorias como string padronizada
saeb_escola[categorical_cols] = saeb_escola[categorical_cols].astype(str).applymap(lambda x: x.strip().upper() if pd.notna(x) else x)

# 5. Substituir valores ausentes
saeb_escola[taxa_cols + media_cols] = saeb_escola[taxa_cols + media_cols].fillna(0)
saeb_escola[categorical_cols] = saeb_escola[categorical_cols].fillna('NÃO TEM')

# 6. Remover duplicatas de escola
saeb_escola = saeb_escola.drop_duplicates(subset=['ID_ESCOLA'], keep='first')

# 7. Remover espaços extras em toda a tabela
saeb_escola = saeb_escola.applymap(lambda x: x.strip() if isinstance(x, str) else x)

# Salvar a base tratada
saeb_escola.to_csv('results/SAEB_escola_tratado.csv', index=False)

print("Base SAEB_escola tratada e pronta para merge com as demais!")


# **limpeza SAEB ALUNO 2EF**

In [ ]:
# 1. Padronizar nomes das colunas
saeb_2ef.columns = saeb_2ef.columns.str.strip().str.upper().str.replace(' ', '_')

# 2. Substituir valores ausentes ou em branco por NaN
saeb_2ef.replace(['', ' ', '.'], np.nan, inplace=True)

# 3. Identificar tipos de colunas
id_cols = [col for col in saeb_2ef.columns if 'ID_' in col or col=='C']
num_cols = [col for col in saeb_2ef.columns if 'PESO' in col or 'PROFICIENCIA' in col or 'ERRO_PADRAO' in col or 'NU_BLOCO' in col or 'IN_' in col]
categorical_cols = [col for col in saeb_2ef.columns if col not in id_cols+num_cols]

# 4. Converter tipos
# IDs como inteiros
saeb_2ef[id_cols] = saeb_2ef[id_cols].astype('Int64')

# Numéricos como float
saeb_2ef[num_cols] = saeb_2ef[num_cols].astype('float')

# Categorias como string padronizada
saeb_2ef[categorical_cols] = saeb_2ef[categorical_cols].astype(str).applymap(lambda x: x.strip().upper() if pd.notna(x) else x)

# 5. Substituir valores ausentes
saeb_2ef[num_cols] = saeb_2ef[num_cols].fillna(0)
saeb_2ef[categorical_cols] = saeb_2ef[categorical_cols].fillna('NÃO TEM')

# 6. Remover duplicatas de aluno
saeb_2ef = saeb_2ef.drop_duplicates(subset=['ID_ALUNO'], keep='first')

# 7. Remover espaços extras em toda a tabela
saeb_2ef = saeb_2ef.applymap(lambda x: x.strip() if isinstance(x, str) else x)

# Salvar a base tratada
saeb_2ef.to_csv('results/SAEB_ALUNO_2EF_tratado.csv', index=False)

print("Base SAEB_2EF tratada e pronta para merge com as demais!")


# **limpeza SAEB ALUNO 5EF**

In [ ]:
# 1. Padronizar nomes das colunas
saeb_5ef.columns = saeb_5ef.columns.str.strip().str.upper().str.replace(' ', '_')

# 2. Substituir valores ausentes ou em branco por NaN
saeb_5ef.replace(['', ' ', '.'], np.nan, inplace=True)

# 3. Identificar tipos de colunas
id_cols = [col for col in saeb_5ef.columns if 'ID_' in col or col=='C']
num_cols = [col for col in saeb_5ef.columns if 'PESO' in col or 'PROFICIENCIA' in col or 'ERRO_PADRAO' in col or 'NU_BLOCO' in col or 'IN_' in col]
categorical_cols = [col for col in saeb_5ef.columns if col not in id_cols+num_cols]

# 4. Converter tipos
# IDs como inteiros
saeb_5ef[id_cols] = saeb_5ef[id_cols].astype('Int64')

# Numéricos como float
saeb_5ef[num_cols] = saeb_5ef[num_cols].astype('float')

# Categorias como string padronizada
saeb_5ef[categorical_cols] = saeb_5ef[categorical_cols].astype(str).applymap(lambda x: x.strip().upper() if pd.notna(x) else x)

# 5. Substituir valores ausentes
saeb_5ef[num_cols] = saeb_5ef[num_cols].fillna(0)
saeb_5ef[categorical_cols] = saeb_5ef[categorical_cols].fillna('NÃO TEM')

# 6. Remover duplicatas de aluno
saeb_5ef = saeb_5ef.drop_duplicates(subset=['ID_ALUNO'], keep='first')

# 7. Remover espaços extras em toda a tabela
saeb_5ef = saeb_5ef.applymap(lambda x: x.strip() if isinstance(x, str) else x)

# Salvar a base tratada
saeb_5ef.to_csv('results/SAEB_ALUNO_5EF_tratado.csv', index=False)

print("Base SAEB_5EF tratada e pronta para merge com as demais!")


# **limpeza SAEB ALUNO 9EF**

In [ ]:
print(f"Base SAEB_9EF carregada. Linhas: {saeb_9ef.shape[0]}, Colunas: {saeb_9ef.shape[1]}")

# Padronização de nomes de colunas
saeb_9ef.columns = (
    saeb_9ef.columns
    .str.strip()              # remove espaços extras
    .str.replace(' ', '_')    # substitui espaços por _
    .str.replace(r'[^\w]', '', regex=True)  # remove caracteres especiais
    .str.upper()              # tudo maiúsculo pra padronizar
)

# Remover espaços em branco em todos os campos de texto
for col in saeb_9ef.select_dtypes(include='object').columns:
    saeb_9ef[col] = saeb_9ef[col].astype(str).str.strip()

# Substituir valores ausentes ou anômalos
saeb_9ef.replace(['.', '..', '...', 'nan', 'NaN', 'NA', ''], np.nan, inplace=True)

# Corrigir possíveis problemas em colunas numéricas
for col in saeb_9ef.columns:
    # tenta converter para número, se possível
    saeb_9ef[col] = pd.to_numeric(saeb_9ef[col], errors='ignore')

# Verificar IDs essenciais e padronizar
id_cols = ['ID_SAEB', 'ID_REGIAO', 'ID_UF', 'ID_MUNICIPIO', 'ID_ESCOLA', 'ID_ALUNO']

for col in id_cols:
    if col in saeb_9ef.columns:
        saeb_9ef[col] = saeb_9ef[col].astype(str).str.replace(r'\.0$', '', regex=True).str.zfill(8).str.strip()

# Corrigir formatações quebradas em campos numéricos (ex: vírgulas, pontos errados)
num_cols = [
    'PROFICIENCIA_LP', 'PROFICIENCIA_MT', 'PROFICIENCIA_CH', 'PROFICIENCIA_CN',
    'ERRO_PADRAO_LP', 'ERRO_PADRAO_MT', 'ERRO_PADRAO_CH', 'ERRO_PADRAO_CN'
]
for col in num_cols:
    if col in saeb_9ef.columns:
        saeb_9ef[col] = (
            saeb_9ef[col]
            .astype(str)
            .str.replace('.', '', regex=False)     # remove pontos de milhares
            .str.replace(',', '.', regex=False)    # troca vírgula decimal por ponto
        )
        saeb_9ef[col] = pd.to_numeric(saeb_9ef[col], errors='coerce')

# Garantir consistência para merges futuros
# Corrige tipos de ID_MUNICIPIO e ID_ESCOLA para bater com Censo
for col in ['ID_MUNICIPIO', 'ID_ESCOLA']:
    if col in saeb_9ef.columns:
        saeb_9ef[col] = saeb_9ef[col].astype(str).str.replace(r'\.0$', '', regex=True).str.strip()

# Remover duplicatas, se existirem
saeb_9ef.drop_duplicates(subset=['ID_ALUNO'], inplace=True)

# Resetar o índice
saeb_9ef.reset_index(drop=True, inplace=True)

# Salvar versão tratada
saeb_9ef.to_csv('results/SAEB_ALUNO_9EF_tratado.csv', index=False)

print("Base SAEB_9EF tratada e salva com sucesso!")
print(f"Formato final: {saeb_9ef.shape[0]} linhas x {saeb_9ef.shape[1]} colunas")

# **limpeza SAEB ALUNO 34EM**

In [ ]:
print(f"Base SAEB_34EM carregada. Linhas: {saeb_34em.shape[0]}, Colunas: {saeb_34em.shape[1]}")

# Padronização de nomes de colunas
saeb_34em.columns = (
    saeb_34em.columns
    .str.strip()
    .str.replace(' ', '_')
    .str.replace(r'[^\w]', '', regex=True)
    .str.upper()
)

# Limpar espaços e caracteres invisíveis em colunas textuais
for col in saeb_34em.select_dtypes(include='object').columns:
    saeb_34em[col] = saeb_34em[col].astype(str).str.strip()

# Substituir valores ausentes ou inválidos
saeb_34em.replace(['.', '..', '...', 'nan', 'NaN', 'NA', ''], np.nan, inplace=True)

# Converter possíveis números com vírgula ou ponto incorreto
num_cols = [
    'PROFICIENCIA_LP', 'ERRO_PADRAO_LP', 'PROFICIENCIA_LP_SAEB', 'ERRO_PADRAO_LP_SAEB',
    'PROFICIENCIA_MT', 'ERRO_PADRAO_MT', 'PROFICIENCIA_MT_SAEB', 'ERRO_PADRAO_MT_SAEB',
    'PESO_ALUNO_LP', 'PESO_ALUNO_MT', 'INSE_ALUNO'
]

for col in num_cols:
    if col in saeb_34em.columns:
        saeb_34em[col] = (
            saeb_34em[col]
            .astype(str)
            .str.replace('.', '', regex=False)   # remove separadores de milhar
            .str.replace(',', '.', regex=False)  # troca vírgula decimal por ponto
        )
        saeb_34em[col] = pd.to_numeric(saeb_34em[col], errors='coerce')

# Padronizar IDs importantes (evita erro de merge com censo ou outras bases)
id_cols = ['ID_SAEB', 'ID_REGIAO', 'ID_UF', 'ID_MUNICIPIO', 'ID_ESCOLA', 'ID_ALUNO']

for col in id_cols:
    if col in saeb_34em.columns:
        saeb_34em[col] = (
            saeb_34em[col]
            .astype(str)
            .str.replace(r'\.0$', '', regex=True)
            .str.strip()
        )

# Corrigir possíveis erros em variáveis categóricas (tipo A, B, C...)
resp_cols = [c for c in saeb_34em.columns if c.startswith('TX_RESP_')]
for col in resp_cols:
    saeb_34em[col] = saeb_34em[col].astype(str).str.upper().str.strip().replace(['NAN', 'NA', '.'], np.nan)

# Remover duplicatas por ID_ALUNO (casos raros)
if 'ID_ALUNO' in saeb_34em.columns:
    saeb_34em.drop_duplicates(subset=['ID_ALUNO'], inplace=True)

# Resetar o índice
saeb_34em.reset_index(drop=True, inplace=True)

# Checagem de consistência básica
print("\nChecando colunas essenciais:")
for col in ['ID_MUNICIPIO', 'ID_ESCOLA', 'PROFICIENCIA_LP', 'PROFICIENCIA_MT']:
    if col in saeb_34em.columns:
        print(f"{col}: {saeb_34em[col].notna().mean():.1%} preenchido")

# Salvar a base tratada
saeb_34em.to_csv('results/SAEB_ALUNO_34EM_tratado.csv', index=False)

print(" Base SAEB_ALUNO_34EM tratada e salva com sucesso!")
print(f"Formato final: {saeb_34em.shape[0]} linhas x {saeb_34em.shape[1]} colunas")


# **UNIÃO DE TODAS AS BASES TRATADAS**

In [ ]:
# Carregar bases já tratadas
base_tea_tdah = pd.read_csv('data/base_tea_tdah_10000.csv')
saeb_2ef = pd.read_csv('data/SAEB_2EF_tratado.csv')
saeb_5ef = pd.read_csv('data/SAEB_ALUNO_5EF_tratado.csv')
saeb_9ef = pd.read_csv('data/SAEB_ALUNO_9EF_tratado.csv')
saeb_34em = pd.read_csv('data/SAEB_ALUNO_34EM_tratado.csv')
saeb_escola = pd.read_csv('data/SAEB_escola_tratado.csv')
censo = pd.read_csv('data/censo_2023_tratado.csv')

print(" Todas as bases carregadas com sucesso!")

# Verificar colunas-chave
print("\nChaves disponíveis nas bases:")
for nome, base in {
    'base_tea_tdah': base_tea_tdah,
    'saeb_2ef': saeb_2ef,
    'saeb_5ef': saeb_5ef,
    'saeb_9ef': saeb_9ef,
    'saeb_34em': saeb_34em,
    'saeb_escola': saeb_escola,
    'censo': censo
}.items():
    print(f"{nome}: {', '.join([c for c in base.columns if 'ID' in c or 'COD' in c])}")




In [ ]:
# 1. Selecionar apenas 9EF e 34EM
saeb_filtrado = pd.concat([saeb_9ef, saeb_34em], ignore_index=True)

# 2. Garantir que SG_UF exista e não seja NaN
# Se não houver SG_UF, você pode mapear a partir de ID_UF ou NO_UF do SAEB
# Vamos assumir que já temos ID_UF ou podemos usar NO_UF -> SG_UF se necessário
saeb_filtrado = saeb_filtrado[saeb_filtrado['ID_UF'].notna()]

# Criar coluna SG_UF a partir de ID_UF (se ID_UF for numérico, transformar em string)
saeb_filtrado['SG_UF'] = saeb_filtrado['ID_UF'].astype(int).map({
    11: 'RO', 12: 'AC', 13: 'AM', 14: 'RR', 15: 'PA', 16: 'AP', 17: 'TO', 21: 'MA',
    22: 'PI', 23: 'CE', 24: 'RN', 25: 'PB', 26: 'PE', 27: 'AL', 28: 'SE', 29: 'BA',
    31: 'MG', 32: 'ES', 33: 'RJ', 35: 'SP', 41: 'PR', 42: 'SC', 43: 'RS', 50: 'MS',
    51: 'MT', 52: 'GO', 53: 'DF'
})



In [ ]:
# 3. Merge com Censo
censo['CO_ENTIDADE'] = censo['CO_ENTIDADE'].astype(str).str.strip()
saeb_filtrado['ID_ESCOLA'] = saeb_filtrado['ID_ESCOLA'].astype(str).str.strip()

base_saeb_censo = saeb_filtrado.merge(
    censo,
    left_on='ID_ESCOLA',
    right_on='CO_ENTIDADE',
    how='left',
    suffixes=('', '_CENSO')
)

# 4. Merge com TEA/TDAH usando SG_UF
base_tea_tdah['UF'] = base_tea_tdah['UF'].astype(str).str.upper().str.strip()
base_final = base_saeb_censo.merge(
    base_tea_tdah,
    left_on='SG_UF',
    right_on='UF',
    how='left',
    suffixes=('', '_TEA_TDAH')
)

print(f"Base final pronta: {base_final.shape}")
print(f"UFs únicas na base final: {base_final['SG_UF'].unique()}")

In [ ]:
# Unir SAEB_alunos com SAEB_escola (chave: ID_ESCOLA)
base_saeb_completa = saeb_alunos_unificado.merge(
    saeb_escola,
    on='ID_ESCOLA',
    how='left',
    suffixes=('', '_ESCOLA')
)

print(f" SAEB + Escola: {base_saeb_completa.shape}")


In [ ]:
# Unir com o CENSO (chave: ID_ESCOLA ou CO_ENTIDADE)
# Primeiro, padronizar colunas de chave se necessário
if 'CO_ENTIDADE' in censo.columns and 'ID_ESCOLA' in base_saeb_completa.columns:
    censo['CO_ENTIDADE'] = censo['CO_ENTIDADE'].astype(str).str.strip()
    base_saeb_completa['ID_ESCOLA'] = base_saeb_completa['ID_ESCOLA'].astype(str).str.strip()

base_saeb_censo = base_saeb_completa.merge(
    censo,
    left_on='ID_ESCOLA',
    right_on='CO_ENTIDADE',
    how='left',
    suffixes=('', '_CENSO')
)

print(f" SAEB + CENSO: {base_saeb_censo.shape}")



In [ ]:
# Normalizar colunas de UF em ambas as bases
base_saeb_censo['SG_UF'] = base_saeb_censo['SG_UF'].astype(str).str.upper().str.strip()
base_tea_tdah['UF'] = base_tea_tdah['UF'].astype(str).str.upper().str.strip()

# Fazer o merge pela sigla da UF
base_final = base_saeb_censo.merge(
    base_tea_tdah,
    left_on='SG_UF',
    right_on='UF',
    how='left',
    suffixes=('', '_TEA_TDAH')
)

print(f" Base final (SAEB + Censo + TEA/TDAH) criada com sucesso! Formato: {base_final.shape}")


# **corrigir Nan em UFs**

In [ ]:
# Caminho do arquivo original
caminho_censo_original = 'data/censo_ed_basica_2023_reduzido.csv'

# Leitura
censo = pd.read_csv(caminho_censo_original, low_memory=False)

print(f"Base Censo original carregada: {censo.shape}")
print("Colunas originais:", censo.columns.tolist()[:10], "...")

In [ ]:
# Padronização de colunas

censo.columns = (
    censo.columns
    .str.strip()
    .str.replace(' ', '_')
    .str.replace('-', '_')
    .str.replace(r'[^\w\s]', '', regex=True)
    .str.upper()
    .map(lambda x: unidecode.unidecode(x))  # remove acentos
)

# Limpeza geral
for col in censo.select_dtypes(include=['object']).columns:
    censo[col] = censo[col].astype(str).str.strip()

# Garantir que UF esteja presente e limpa
if 'NO_UF' in censo.columns and 'SG_UF' in censo.columns:
    censo['NO_UF'] = censo['NO_UF'].astype(str).str.strip().str.upper()
    censo['SG_UF'] = censo['SG_UF'].astype(str).str.strip().str.upper()
else:
    print(" Atenção: Colunas NO_UF/SG_UF não encontradas na base original.")

# Checagem de nulos em UF
print("\nVerificação de UF:")
print(censo[['NO_UF', 'SG_UF']].drop_duplicates().head(15))

# Exportar base tratada
caminho_tratado = 'results/censo_2023_tratado.csv'
censo.to_csv(caminho_tratado, index=False, encoding='utf-8-sig')

print(f"\n Base tratada salva em: {caminho_tratado}")

In [ ]:
# Verificação rápida
print("\nResumo da base final:")
print(base_final[['UF', 'ID_ESCOLA', 'PROFICIENCIA_LP', 'PROFICIENCIA_MT', 'TDAH', 'TEA']].head())

#  Salvar resultado
base_final.to_csv('results/base_final_integrada.csv', index=False)
print("\n Base final salva como 'base_final_integrada.csv'")

teste


In [ ]:
base_saeb_completa['ID_UF'] = base_saeb_completa['ID_UF'].astype('Int64').astype(str)


In [ ]:
uf_dict = {
    '11':'RO','12':'AC','13':'AM','14':'RR','15':'PA','16':'AP','17':'TO',
    '21':'MA','22':'PI','23':'CE','24':'RN','25':'PB','26':'PE','27':'AL',
    '28':'SE','29':'BA','31':'MG','32':'ES','33':'RJ','35':'SP',
    '41':'PR','42':'SC','43':'RS','50':'MS','51':'MT','52':'GO','53':'DF'
}


In [ ]:
base_saeb_completa['SG_UF'] = base_saeb_completa['ID_UF'].map(uf_dict)


In [ ]:
print(base_saeb_completa['SG_UF'].unique())
print(base_saeb_completa['SG_UF'].isna().sum())
